In [1]:
import json
import pandas as pd
from pathlib import Path

In [3]:
raw_data_path = Path("diverse_summ.json")
excel_path = raw_data_path.with_suffix(".xlsx")
MAX_CELL_LENGTH = 32767
TRUNCATION_NOTE = "\n[Truncated to Excel's character limit]"
def fit_excel_cell(value):
    if value is None:
        return ""
    text = str(value)
    if len(text) > MAX_CELL_LENGTH:
        allowed_length = MAX_CELL_LENGTH - len(TRUNCATION_NOTE)
        return text[:allowed_length] + TRUNCATION_NOTE
    return text
with open(raw_data_path, "r", encoding="utf-8") as file:
    data = json.load(file)
rows = []
for record in data:
    eid = record.get("eid", "")
    # Match each answer's aid with its corresponding article
    articles_by_id = {
        article.get("aid"): article
        for article in record.get("articles", [])
    }
    for question_item in record.get("question_answers", []):
        question = question_item.get("question", "")
        for answer_group in question_item.get("answer_groups", []):
            for answer_item in answer_group:
                answer_aid = answer_item.get("aid", "")
                article = articles_by_id.get(answer_aid, {})
                rows.append({
                    "eid": fit_excel_cell(eid),
                    "article_aid": fit_excel_cell(article.get("aid", "")),
                    "article_title": fit_excel_cell(article.get("title", "")),
                    "article_url": fit_excel_cell(article.get("url", "")),
                    "article_domain": fit_excel_cell(article.get("domain", "")),
                    "article_content": fit_excel_cell(article.get("content", "")),
                    "question": fit_excel_cell(question),
                    "answer": fit_excel_cell(answer_item.get("answer", "")),
                    "answer_aid": fit_excel_cell(answer_aid)
                })
df = pd.DataFrame(rows)
# Export to Excel
df.to_excel(excel_path, index=False, sheet_name="Flattened Data")
print(f"Excel file saved to: {excel_path}")
print(f"Number of rows: {len(df)}")

Excel file saved to: diverse_summ.xlsx
Number of rows: 6561


In [6]:
# check missing values
print(df.isnull().sum())

eid                0
article_aid        0
article_title      0
article_url        0
article_domain     0
article_content    0
question           0
answer             0
answer_aid         0
dtype: int64


In [10]:
#make data readable
input_file = Path("diverse_summ.xlsx")
output_file = input_file.with_name(input_file.stem + "_extracted.xlsx")
#required attributes
EID_COLUMN = "eid"
AID_COLUMN = "article_aid"
TITLE_COLUMN = "article_title"
WEBSITE_COLUMN = "article_url"
# assign sequential numbers to EID and AID columns
def create_sequential_ids(column):
    id_mapping = {}
    sequential_values = []
    next_id = 1
    for value in column:
        if pd.isna(value) or str(value).strip() == "":
            sequential_values.append(pd.NA)
            continue
        original_id = str(value).strip()
        if original_id not in id_mapping:
            id_mapping[original_id] = next_id
            next_id += 1
        sequential_values.append(id_mapping[original_id])
    return pd.Series(sequential_values, dtype="Int64")
# Create independent sequential IDs
df[EID_COLUMN] = create_sequential_ids(df[EID_COLUMN])
# AID numbering continues throughout the entire dataset
df[AID_COLUMN] = create_sequential_ids(df[AID_COLUMN])
# Keep only the required four columns
final_df = df[[
        EID_COLUMN,
        AID_COLUMN,
        TITLE_COLUMN,
        WEBSITE_COLUMN
    ]].copy()
# Rename columns for the output
final_df.columns = [
    "Event ID",
    "Reference Article ID",
    "Reference Article Title",
    "Website"
]
# Save the result
final_df.to_excel(output_file,index=False,sheet_name="Reference Articles")
# Remove completely duplicated rows
final_df = final_df.drop_duplicates().reset_index(drop=True)
# Save the cleaned data
final_df.to_excel(
    output_file,
    index=False,
    sheet_name="Reference Articles"
)
print(f"File saved to: {output_file}")
print(f"Rows after removing duplicates: {len(final_df)}")
print(f"Unique Event IDs: {final_df['Event ID'].nunique()}")
print(
    "Unique Reference Article IDs:",
    final_df["Reference Article ID"].nunique()
)


File saved to: diverse_summ_extracted.xlsx
Rows after removing duplicates: 1839
Unique Event IDs: 245
Unique Reference Article IDs: 1743
